# AfroLuo — MarianMT Français → Ngiemboon (version finale, 14 nov. 2026)

Base : `Helsinki-NLP/opus-mt-fr-en` pour FR→NNH (et `opus-mt-en-fr` pour NNH→FR, facultatif).

**Données** (dans `afroluo_data_nnh.zip`, construites dans le notebook) :
- corpus général (Bible MINBA, dictionnaires, corpus parallèles) ;
- `dico_complet` (10 191 entrées, orthographe alignée sur le corpus : w→ẅ, n→ŋ quand le corpus l'atteste) ;
- corpus financier : lignes validées ×10, brouillons Claude ×3.

**Ajouts par rapport à l'ancien notebook :**
1. Les jetons `{MONTANT}`, `{NOM}`… sont remplacés par des nombres (9001, 9112…) **dès l'entraînement**, avec exactement la même règle qu'au test. Avant, le modèle apprenait `{MONTANT}` mais recevait `9001` au test.
2. Le sens retour NNH→FR (aller-retour) est **facultatif** : l'ancien notebook le chargeait sans l'avoir entraîné, et plantait.
3. Si vous avez le fichier `prioritaire_275_a_verifier.csv` corrigé par le locuteur, téléversez-le avec le zip : ses lignes remplacent mes brouillons et passent en « validé ».
4. Colonne « mémoire » au test final : la traduction exacte du corpus quand la phrase y existe, à utiliser en priorité dans l'extension.

**Avant de lancer :** Exécution → Modifier le type d'exécution → GPU T4. Ne téléversez **pas** de CSV ré-enregistré par Excel (Excel abîme ɔ, ŋ, ʉ, ẅ).

In [1]:
!pip install evaluate sacrebleu sentencepiece sacremoses -q
from google.colab import drive, files
drive.mount('/content/drive')
import os
DRIVE_BASE = "/content/drive/MyDrive/afroluo_models_nnh/"
os.makedirs(DRIVE_BASE, exist_ok=True)
ALLER_RETOUR = False   # True = entraîne aussi NNH→FR (≈ 2× plus long) pour le contrôle par aller-retour
print("Prêt !")

Mounted at /content/drive
Prêt !


In [2]:
# Sélectionnez afroluo_data_nnh.zip (et, si vous l'avez, prioritaire_275_a_verifier.csv corrigé)
uploaded = files.upload()
!unzip -q -o afroluo_data_nnh.zip -d /content/
DATA_DIR = "/content/afroluo_data_nnh/"
import sys, glob
sys.path.insert(0, DATA_DIR)
from construire import construire, masquer
f275 = [f for f in uploaded if f.startswith("prioritaire_275")]
construire(f"/content/{f275[0]}" if f275 else None)
!wc -l /content/afroluo_data_nnh/fr_nnh/*.jsonl

Saving afroluo_data_nnh.zip to afroluo_data_nnh.zip
Saving prioritaire_275_a_verifier.csv to prioritaire_275_a_verifier.csv
train=61162 | validation=627 | test=534 (dont 34 phrases financières validées jamais vues)
  corpus:corpus_ngiemboon_de_MINBA_hUggingface  41715
  corpus:corpus_parallele2                      679
  corpus:corpus_parallele4                      1967
  corpus:dict0_ngiemboon-francais-anglais       1225
  corpus:dictionnaire2_ligne_par_ligne_mot      6227
  dico_complet                                  9802
  fin_brouillons                                2256
  fin_exclu                                     29
  fin_valides                                   353
  prioritaire_275_corrige                       275
     534 /content/afroluo_data_nnh/fr_nnh/test.jsonl
   61162 /content/afroluo_data_nnh/fr_nnh/train.jsonl
     627 /content/afroluo_data_nnh/fr_nnh/validation.jsonl
   62323 total


In [3]:
import json, unicodedata, re
from translit import encoder_nnh, decoder_nnh

def load_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]
train_raw = load_jsonl(f"{DATA_DIR}/fr_nnh/train.jsonl")
nnh = [x["translation"]["nnh"] for x in train_raw]
corrompu = sum(bool(re.search("Ã|Ê|Å|â€", s)) for s in nnh)
perte = sum(decoder_nnh(encoder_nnh(s)) != s for s in nnh)
print(len(train_raw), "paires | lignes corrompues :", corrompu, "(doit être 0) | pertes translittération :", perte, "(doit être 0)")
for x in train_raw[:3]: print(x["translation"])

61162 paires | lignes corrompues : 0 (doit être 0) | pertes translittération : 0 (doit être 0)
{'fr': 'Leurs gens sont depuis venus, où sont alors les nôtres ?', 'nnh': "Epɔ́b pʉ̀a ně ndá' ńtó, epégè gwɔ́ pɔ́ nzɔ̌?"}
{'fr': 'A présent, vous connaissez Dieu. Bien plus : Dieu vous a reconnus comme siens. Comment se peut-il alors que vous retourniez à ces principes élémentaires sans pouvoir ni valeur, pour en devenir à nouveau les esclaves ?', 'nnh': "Á piŋ ńgʉa na fʉ̀'ɔɔn pi ne ńzsé Ssé tà mèŋ kɔ'ɔ́ fó gɔɔn pɔ́ ngie Ssé ne ńzsé gẅí. Á piŋ ngǎ pi gíŋe ḿbiŋe nzèm, á tsɛ̀ɛ sɛ̌ shúm sie é té metʉ̀ gẅiin, ńké te tsɔ́ mmó gẅiin? Pi kwoŋo légíŋ ḿbiŋ nzèm ńzẅíŋ menkwɔɔn mbwó pɔ́b keen?"}
{'fr': 'Tout ce qui, en Israël, m’a été voué définitivement sera pour toi .', 'nnh': "Akɔ̀ɔkɔ́ɔ wɔ́ɔ gie pa Ésàlayɛ něŋ nzse mǒ, ńgwá' ngie áa ye Cʉ̀apʉ̀a, á ge gÿo gwɔ́ pɔ́ gí."}


In [4]:
from transformers import MarianTokenizer
import random
random.seed(0); ech = random.sample(nnh, 2000)
t_fr_en = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-fr-en")   # côté cible pour FR→NNH
t_en_fr = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-fr")   # côté source pour NNH→FR

def taux_inconnus(tok, textes, cible):
    ids = [i for s in textes for i in (tok(text_target=s).input_ids if cible else tok(s).input_ids)]
    return ids.count(tok.unk_token_id) / len(ids)

brut_c = taux_inconnus(t_fr_en, ech, True); brut_s = taux_inconnus(t_en_fr, ech, False)
tr = [encoder_nnh(s) for s in ech]
tr_c = taux_inconnus(t_fr_en, tr, True); tr_s = taux_inconnus(t_en_fr, tr, False)
print(f"Sans translittération : {brut_c:.2%} (FR→NNH) | {brut_s:.2%} (NNH→FR) de jetons inconnus")
print(f"Avec translittération : {tr_c:.2%} (FR→NNH) | {tr_s:.2%} (NNH→FR)")
TRANSLIT = max(brut_c, brut_s) > 0.005
print("=> Translittération", "ACTIVÉE" if TRANSLIT else "inutile", "| exemple :", encoder_nnh(ech[0])[:120])
# Contrôle : une phrase type doit revenir intacte après tokenisation + décodage
s = "pfwɔ̀' nkáb yembǐ gẅi ŋ́gwɔ́ 9001 FCFA."
enc = encoder_nnh(s) if TRANSLIT else s
dec = t_fr_en.decode(t_fr_en(text_target=enc).input_ids, skip_special_tokens=True)
print("Aller-retour tokenizer :", (decoder_nnh(dec) if TRANSLIT else dec) == s, "|", dec)

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

Sans translittération : 0.05% (FR→NNH) | 0.05% (NNH→FR) de jetons inconnus
Avec translittération : 0.00% (FR→NNH) | 0.00% (NNH→FR)
=> Translittération inutile | exemple : Tqute tqute, pege lqogqo shyqdqoqlg nqhdyqdqoqh', nqhkwqdeqh pquqla peqhgeql, mbqoqo metyqdoqvsoqno Sseqh yege. Taqh', a
Aller-retour tokenizer : True | pfwɔ̀' nkáb yembǐ gẅi ŋ́gwɔ́ 9001 FCFA.


In [5]:
import numpy as np, shutil
from datasets import Dataset, DatasetDict
from transformers import (MarianMTModel, Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          DataCollatorForSeq2Seq, EarlyStoppingCallback, TrainerCallback)
import evaluate

BASE_MODELS = {"fr_nnh": "Helsinki-NLP/opus-mt-fr-en", "nnh_fr": "Helsinki-NLP/opus-mt-en-fr"}
OUTPUT_BASE = "/content/models_nnh/"
EPOCHS = 10          # arrêt anticipé (patience 2) de toute façon
MAX_LEN = 200        # la translittération allonge le Ngiemboon : 128 tronquait les longs versets

def prep_nnh(s):  return encoder_nnh(s) if TRANSLIT else s
def post_nnh(s):  return decoder_nnh(s) if TRANSLIT else s

class SaveToDriveCallback(TrainerCallback):
    def __init__(self, drive_path, save_every=2):
        self.drive_path, self.save_every = drive_path, save_every
    def on_epoch_end(self, args, state, control, **kwargs):
        e = int(state.epoch)
        if e % self.save_every == 0:
            shutil.copytree(args.output_dir, f"{self.drive_path}/checkpoint_epoch_{e}", dirs_exist_ok=True)
            print(f"\nSauvegarde Drive epoch {e}")

def load_dataset_dir(direction):
    src, tgt = direction.split("_")
    out = {}
    for split in ["train", "validation", "test"]:
        raw = load_jsonl(f"{DATA_DIR}/{direction}/{split}.jsonl")
        s = [x["translation"][src] for x in raw]; t = [x["translation"][tgt] for x in raw]
        if src == "nnh": s = [prep_nnh(x) for x in s]
        if tgt == "nnh": t = [prep_nnh(x) for x in t]
        out[split] = Dataset.from_dict({"source": s, "target": t}); print(f"  {split}: {len(s)}")
    return DatasetDict(out)

def preprocess(ex, tok):
    m = tok(ex["source"], max_length=MAX_LEN, truncation=True)
    m["labels"] = tok(text_target=ex["target"], max_length=MAX_LEN, truncation=True)["input_ids"]
    return m

bleu_metric = evaluate.load("sacrebleu"); chrf_metric = evaluate.load("chrf")
def compute_metrics(p, tok, direction):
    preds, labels = p
    preds = np.where(preds != -100, preds, tok.pad_token_id)
    dp = tok.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tok.pad_token_id)
    dl = tok.batch_decode(labels, skip_special_tokens=True)
    if direction.endswith("nnh"):
        dp = [post_nnh(x) for x in dp]; dl = [post_nnh(x) for x in dl]
    refs = [[l.strip()] for l in dl]; hyps = [x.strip() for x in dp]
    return {"bleu": round(bleu_metric.compute(predictions=hyps, references=refs)["score"], 2),
            "chrf": round(chrf_metric.compute(predictions=hyps, references=refs)["score"], 2)}

def train_direction(direction):
    print(f"\n{'='*60}\nENTRAINEMENT : {direction.upper()}\n{'='*60}")
    base = BASE_MODELS[direction]
    out_dir = os.path.join(OUTPUT_BASE, f"marianmt_{direction}")
    drive_path = os.path.join(DRIVE_BASE, f"marianmt_{direction}"); os.makedirs(drive_path, exist_ok=True)
    tok = MarianTokenizer.from_pretrained(base); model = MarianMTModel.from_pretrained(base)
    ds = load_dataset_dir(direction)
    tokd = ds.map(lambda x: preprocess(x, tok), batched=True, remove_columns=["source", "target"])
    args = Seq2SeqTrainingArguments(
        output_dir=out_dir, num_train_epochs=EPOCHS,
        per_device_train_batch_size=16, per_device_eval_batch_size=32,
        warmup_steps=500, weight_decay=0.01, logging_steps=200,
        eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model="chrf", greater_is_better=True,
        predict_with_generate=True, generation_max_length=MAX_LEN, generation_num_beams=1,
        fp16=True, learning_rate=5e-5, save_total_limit=2, report_to="none")
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=tokd["train"], eval_dataset=tokd["validation"],
        processing_class=tok, data_collator=DataCollatorForSeq2Seq(tokenizer=tok, model=model, padding=True),
        compute_metrics=lambda p: compute_metrics(p, tok, direction),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2), SaveToDriveCallback(drive_path, save_every=2)])
    trainer.train()
    res = trainer.evaluate(tokd["test"]); print(f"TEST {direction} : BLEU {res.get('eval_bleu')} | chrF {res.get('eval_chrf')}")
    trainer.save_model(out_dir); tok.save_pretrained(out_dir)
    shutil.copytree(out_dir, f"{drive_path}/final", dirs_exist_ok=True)
    shutil.copy(f"{DATA_DIR}/translit.py", f"{drive_path}/final/translit.py")
    with open(f"{drive_path}/final/translit.json", "w") as f: json.dump({"TRANSLIT": TRANSLIT}, f)
    with open(f"{drive_path}/test_metrics.json", "w") as f: json.dump(res, f, indent=2)
    return res
print("Script chargé et prêt !")

Script chargé et prêt !


In [6]:
resultats = {"fr_nnh": train_direction("fr_nnh")}
if ALLER_RETOUR:
    resultats["nnh_fr"] = train_direction("nnh_fr")
resultats


ENTRAINEMENT : FR_NNH


config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  301MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

... (21 outputs left out)

Training Loss,Validation Loss,Epoch,Bleu,Chrf
0.658462,0.871372,10,18.860000,40.790000


TEST fr_nnh : BLEU 18.86 | chrF 40.79


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'fr_nnh': {'eval_loss': 0.8713721036911011,
  'eval_bleu': 18.86,
  'eval_chrf': 40.79}}

In [7]:
TEST = [
"Solde",
"Compte",
"Montant",
"Frais",
"Transfert",
"Envoyer de l'argent",
"Recevoir de l'argent",
"Bénéficiaire",
"Expéditeur",
"Numéro de téléphone",
"Code PIN",
"Mot de passe",
"Code de vérification",
"Confirmer",
"Annuler",
"Retour",
"Suivant",
"Valider",
"Total",
"Récapitulatif",
"Transaction",
"Référence",
"Date",
"Disponible",
"Insuffisant",
"Réussi",
"Échec",
"En attente",
"Erreur",
"Attention",
"Sécurité",
"Fraude",
"Arnaque",
"Retrait",
"Dépôt",
"Paiement",
"Payer une facture",
"Historique des transactions",
"Reçu",
"Commission",
"Taxe",
"Plafond",
"Agent",
"Épargne",
"Prêt",
"Taux d'intérêt",
"Échéance",
"Service client",
"Accueil",
"Paramètres",
"Aide",
"Se connecter",
"Se déconnecter",
"Écouter",
"Voir le texte original",
"Bienvenue",
"Bonjour {NOM}",
"Que voulez-vous faire ?",
"Choisissez une option",
"Appuyez sur le bouton pour continuer",
"Veuillez patienter",
"Chargement en cours",
"Ce champ est obligatoire",
"Étape {N} sur {TOTAL}",
"Retour à l'accueil",
"Consulter mon solde",
"Votre solde est de {MONTANT}",
"Solde disponible : {MONTANT}",
"Afficher le solde",
"Masquer le solde",
"Dernière mise à jour : {DATE}",
"Votre solde a été mis à jour",
"Saisissez votre code PIN pour voir votre solde",
"Frais de transfert : {FRAIS}",
"Frais de l'opération : {FRAIS}",
"Les frais seront déduits de votre solde",
"Aucuns frais pour cette opération",
"Montant total à payer : {MONTANT}",
"Les frais dépendent du montant envoyé",
"Voir la grille tarifaire",
"Calculer les frais",
"Envoyer de l'argent à quelqu'un",
"Saisissez le numéro de téléphone du bénéficiaire",
"Saisissez le montant à envoyer",
"Choisissez un bénéficiaire",
"Ajouter un nouveau bénéficiaire",
"Nom du bénéficiaire",
"Motif du transfert (facultatif)",
"Vous allez envoyer {MONTANT} à {NOM}",
"Numéro du bénéficiaire : {NUMERO}",
"Choisir dans mes contacts",
"Enregistrer ce bénéficiaire",
"Vérifiez les informations avant de confirmer",
"Bénéficiaire : {NOM}",
"Montant envoyé : {MONTANT}",
"Frais : {FRAIS}",
"Total débité de votre compte : {MONTANT}",
"Le bénéficiaire recevra : {MONTANT}",
"Saisissez votre code PIN pour confirmer",
"Cette opération ne peut pas être annulée après confirmation",
"Voulez-vous vraiment confirmer cette opération ?",
"Modifier",
"Transfert réussi",
"Votre transfert de {MONTANT} à {NOM} a été effectué",
"Référence de la transaction : {REF}",
"Nouveau solde : {MONTANT}",
"Opération en cours de traitement",
"Opération échouée",
"Télécharger le reçu",
"Partager le reçu",
"Solde insuffisant",
"Votre solde est insuffisant pour effectuer cette opération",
"Code PIN incorrect",
"Il vous reste {N} tentatives",
"Numéro de téléphone invalide",
"Le montant minimum est de {MONTANT}",
"Le montant maximum est de {MONTANT}",
"Vous avez dépassé votre plafond journalier",
"Le service est temporairement indisponible. Réessayez plus tard.",
"Votre session a expiré. Veuillez vous reconnecter.",
"Votre compte est bloqué. Contactez le service client.",
"Une erreur est survenue. Veuillez réessayer.",
"Vérifiez votre connexion internet",
"Ce bénéficiaire n'existe pas",
"Ne communiquez jamais votre code PIN à personne",
"Aucun agent ne vous demandera jamais votre code PIN",
"Ne partagez jamais votre code de vérification",
"Vérifiez toujours le nom du bénéficiaire avant de confirmer",
"Méfiez-vous des messages qui vous demandent d'envoyer de l'argent",
"Si vous n'êtes pas à l'origine de cette opération, contactez immédiatement le service client",
"Attention : tentative de fraude possible",
"Ne cliquez pas sur des liens que vous ne connaissez pas",
"Un code vous a été envoyé par SMS au {NUMERO}",
"Personne ne doit voir votre écran quand vous saisissez votre code",
"Message de sécurité important. Lisez le texte original ou faites-vous aider par une personne de confiance.",
"Traduction automatique : vérifiez avec le texte original",
"Ce texte n'a pas pu être traduit de façon sûre. Le texte original est affiché.",
"Les montants et les noms ne sont jamais modifiés par la traduction",
"Choisissez votre langue",
"Le solde, c'est l'argent qui se trouve sur votre compte.",
"Les frais, c'est l'argent que le service prend pour faire l'opération.",
"Le bénéficiaire, c'est la personne qui va recevoir l'argent.",
"Avant de confirmer, vérifiez le nom, le montant et les frais.",
"Votre code PIN est secret. Ne le donnez à personne, même pas à un agent.",
"Si quelque chose vous semble bizarre, arrêtez et demandez de l'aide.",
"Le total débité, c'est le montant envoyé plus les frais."
]

In [8]:
import torch, pandas as pd, sacrebleu, csv
dev = "cuda" if torch.cuda.is_available() else "cpu"
def charger(direction):
    p = f"{DRIVE_BASE}/marianmt_{direction}/final"
    return MarianTokenizer.from_pretrained(p), MarianMTModel.from_pretrained(p).to(dev).eval()
tk1, m1 = charger("fr_nnh")
retour_ok = ALLER_RETOUR and os.path.isdir(f"{DRIVE_BASE}/marianmt_nnh_fr/final")
if retour_ok: tk2, m2 = charger("nnh_fr")

def gen(tok, mod, textes, lot=16):
    out = []
    for i in range(0, len(textes), lot):
        enc = tok(textes[i:i+lot], return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(dev)
        with torch.no_grad(): g = mod.generate(**enc, num_beams=4, max_length=MAX_LEN)
        out += tok.batch_decode(g, skip_special_tokens=True)
    return out

# Mémoire de traduction : phrases exactes du corpus financier (prioritaires dans l'extension)
memoire = {}
for r in csv.DictReader(open(f"{DATA_DIR}/sources/corpus_financier_ngiemboon_complet.csv", encoding="utf-8-sig"), delimiter=";"):
    if not r["statut"].startswith("valeur"): memoire[r["francais"].strip().lower()] = (r["ngiemboon"], r["statut"])

JETON = re.compile(r"\{[A-Z_]+\}")
masq = [masquer(p) for p in TEST]                       # même règle qu'à l'entraînement
brut = [post_nnh(s) for s in gen(tk1, m1, [a for a, _, _ in masq])]
fin, ok = [], []
for src, (_, _, t), s in zip(TEST, masq, brut):
    for j, c in t.items(): s = s.replace(c, j)
    good = sorted(set(JETON.findall(src))) == sorted(set(JETON.findall(s))) and not re.search(r"9\d{3}", s)
    ok.append(good); fin.append(s)
res = pd.DataFrame({"francais": TEST, "ngiemboon_modele": fin, "jetons_ok": ok})
res["memoire"] = [memoire.get(p.strip().lower(), ("", ""))[0] for p in TEST]
res["statut_memoire"] = [memoire.get(p.strip().lower(), ("", ""))[1] for p in TEST]
res["affichage_final"] = [m if m else (s if g else f) for m, s, g, f in zip(res.memoire, fin, ok, TEST)]
if retour_ok:
    res["retour_francais"] = gen(tk2, m2, [prep_nnh(s) for s in fin])
    res["chrF_aller_retour"] = [sacrebleu.sentence_chrf(h, [s]).score for h, s in zip(res.retour_francais, res.francais)]
    res["suspect"] = res.chrF_aller_retour < 40
avec = res.francais.str.contains(r"\{")
print(f"Montants/noms préservés : {res.loc[avec,'jetons_ok'].mean():.0%} | phrases trouvées en mémoire : {(res.memoire!='').sum()}/{len(res)}")
res["verdict_locuteur"] = ""; res["correction"] = ""
res.to_excel(f"{DRIVE_BASE}/resultats_safinum_ngiemboon_marian.xlsx", index=False)   # pour lecture seulement
res.to_csv(f"{DRIVE_BASE}/resultats_safinum_ngiemboon_marian.csv", index=False, encoding="utf-8-sig")
files.download(f"{DRIVE_BASE}/resultats_safinum_ngiemboon_marian.xlsx")
res.head(30)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Montants/noms préservés : 100% | phrases trouvées en mémoire : 146/146


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,francais,ngiemboon_modele,jetons_ok,memoire,statut_memoire,affichage_final,verdict_locuteur,correction
0,Solde,pfwɔ̀' nkáb,True,pfwɔ̀' nkáb,glossaire validé,pfwɔ̀' nkáb,,
1,Compte,compte,True,compte,brouillon Claude – à vérifier,compte,,
2,Montant,nzẅìŋ nkáb,True,nzẅìŋ nkáb,glossaire validé,nzẅìŋ nkáb,,
3,Frais,mekǔ,True,mekǔ,glossaire validé,mekǔ,,
4,Transfert,leńtúm,True,leńtúm,brouillon Claude – à vérifier,leńtúm,,
5,Envoyer de l'argent,ńtúm nkáb,True,ńtúm nkáb,brouillon Claude – à vérifier,ńtúm nkáb,,
6,Recevoir de l'argent,ńkúe nkáb,True,ńkúe nkáb,brouillon Claude – à vérifier,ńkúe nkáb,,
7,Bénéficiaire,ngàŋ nkʉ̀,True,ngàŋ nkʉ̀,glossaire validé,ngàŋ nkʉ̀,,
8,Expéditeur,ŋwè'e nkʉ̀,True,ŋwè'e nkʉ̀,brouillon Claude – à vérifier,ŋwè'e nkʉ̀,,
9,Numéro de téléphone,number kẅíkẅíle,True,number kẅíkẅíle,glossaire validé,number kẅíkẅíle,,


## Export ONNX (pour l'extension)
Dans l'extension, appliquez **la même règle de masquage** : remplacer chaque valeur protégée (montant, nom, numéro, référence) par 9001, 9112, 9223… avant la traduction, puis la remettre ensuite. Le modèle a appris à recopier ces nombres. Si `TRANSLIT` est vrai, décodez la sortie avec `decoder_nnh` (`translit.py` est copié à côté du modèle).

In [11]:
!pip uninstall -y -q diffusers
!pip list 2>/dev/null | grep -iE "^(transformers|huggingface-hub|optimum|optimum-onnx|onnxruntime|torch) "

import os, glob, shutil, subprocess
SRC = f"{DRIVE_BASE}/marianmt_fr_nnh/final"
OUT = "/content/onnx_fr_nnh"
shutil.rmtree(OUT, ignore_errors=True)
if os.path.exists(f"{DRIVE_BASE}/onnx_fr_nnh.zip"): os.remove(f"{DRIVE_BASE}/onnx_fr_nnh.zip")   # le zip vide

r = subprocess.run(["optimum-cli", "export", "onnx", "--model", SRC,
                    "--task", "text2text-generation-with-past", OUT], capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-3000:])
fichiers = glob.glob(f"{OUT}/*.onnx")
assert fichiers, "EXPORT ÉCHOUÉ : rien n'est produit, ne pas continuer (copiez-moi l'erreur ci-dessus)"
print("Exportés :", [os.path.basename(f) for f in fichiers])

from onnxruntime.quantization import quantize_dynamic, QuantType
os.makedirs(f"{OUT}/onnx", exist_ok=True)
for f in fichiers:
    nom = os.path.basename(f)
    shutil.move(f, f"{OUT}/onnx/{nom}")
    quantize_dynamic(f"{OUT}/onnx/{nom}", f"{OUT}/onnx/{nom[:-5]}_quantized.onnx", weight_type=QuantType.QInt8)
shutil.copy(f"{DATA_DIR}/translit.py", OUT)
!du -sh {OUT}/onnx/*
!cd /content && zip -qr onnx_fr_nnh.zip onnx_fr_nnh && cp onnx_fr_nnh.zip {DRIVE_BASE}
print("OK :", DRIVE_BASE + "onnx_fr_nnh.zip")

onnxruntime                           1.32.0
optimum                               2.1.0
optimum-onnx                          0.1.0
torch                                 2.14.1+cu130
transformers                          4.57.6
 sformers/integrations/sdpa_attention.py:81: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  is_causal = query.shape[2] > 1 and attention_mask is None and getattr(module, "is_causal", True)
/usr/local/lib/python3.13/dist-packages/transformers/models/marian/modeling_marian.py:642: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if s

215M	/content/onnx_fr_nnh/onnx/decoder_model_merged.onnx
215M	/content/onnx_fr_nnh/onnx/decoder_model_merged_quantized.onnx
214M	/content/onnx_fr_nnh/onnx/decoder_model.onnx
55M	/content/onnx_fr_nnh/onnx/decoder_model_quantized.onnx
202M	/content/onnx_fr_nnh/onnx/decoder_with_past_model.onnx
52M	/content/onnx_fr_nnh/onnx/decoder_with_past_model_quantized.onnx
190M	/content/onnx_fr_nnh/onnx/encoder_model.onnx
48M	/content/onnx_fr_nnh/onnx/encoder_model_quantized.onnx
OK : /content/drive/MyDrive/afroluo_models_nnh/onnx_fr_nnh.zip


In [12]:
from onnxruntime.quantization import quantize_dynamic, QuantType
D = "/content/onnx_fr_nnh/onnx"
quantize_dynamic(f"{D}/decoder_model_merged.onnx", f"{D}/decoder_model_merged_quantized.onnx",
                 weight_type=QuantType.QInt8, extra_options={"EnableSubgraph": True})
!du -sh {D}/decoder_model_merged_quantized.onnx

55M	/content/onnx_fr_nnh/onnx/decoder_model_merged_quantized.onnx


In [16]:
import os, json, re, sys, numpy as np, onnxruntime as ort
from transformers import MarianTokenizer, MarianMTModel
FINAL = "/content/drive/MyDrive/afroluo_models_nnh/marianmt_fr_nnh/final"
Q = "/content/onnx_fr_nnh/onnx"
sys.path.insert(0, FINAL)
from translit import decoder_nnh
TRANSLIT = json.load(open(f"{FINAL}/translit.json"))["TRANSLIT"]
post = decoder_nnh if TRANSLIT else (lambda s: s)
tok = MarianTokenizer.from_pretrained(FINAL)
pt = MarianMTModel.from_pretrained(FINAL).eval()
cfg = pt.config

def masquer(p):
    t = {}
    for j in re.findall(r"\{[A-Z_]+\}", p): t.setdefault(j, str(9001 + 111 * len(t)))
    for j, c in t.items(): p = p.replace(j, c)
    return p

phrases = ["Consulter mon solde", "Votre solde est de {MONTANT}", "Frais de transfert : {FRAIS}",
           "Saisissez le numéro de téléphone du bénéficiaire", "Vous allez envoyer {MONTANT} à {NOM}",
           "Confirmer", "Annuler", "Transfert réussi", "Solde insuffisant", "Code PIN incorrect",
           "Ne communiquez jamais votre code PIN à personne", "Une erreur est survenue. Veuillez réessayer.",
           "Votre compte est bloqué. Contactez le service client.", "Nouveau solde : {MONTANT}",
           "Le bénéficiaire recevra : {MONTANT}", "Vérifiez les informations avant de confirmer",
           "Opération échouée", "Télécharger le reçu", "Attention : tentative de fraude possible",
           "Total débité de votre compte : {MONTANT}"]
src = [masquer(p) for p in phrases]

enc_s = ort.InferenceSession(f"{Q}/encoder_model_quantized.onnx", providers=["CPUExecutionProvider"])
dec_s = ort.InferenceSession(f"{Q}/decoder_model_quantized.onnx", providers=["CPUExecutionProvider"])
enc_in = {i.name for i in enc_s.get_inputs()}; dec_in = {i.name for i in dec_s.get_inputs()}
print("Entrées encodeur :", enc_in, "| décodeur :", dec_in)

def onnx_glouton(texte, max_len=200):
    e = tok([texte], return_tensors="np")
    ids, am = e["input_ids"].astype(np.int64), e["attention_mask"].astype(np.int64)
    h = enc_s.run(None, {k: v for k, v in {"input_ids": ids, "attention_mask": am}.items() if k in enc_in})[0]
    out = [cfg.decoder_start_token_id]
    for _ in range(max_len):
        feeds = {"input_ids": np.array([out], dtype=np.int64), "encoder_hidden_states": h,
                 "encoder_attention_mask": am}
        logits = dec_s.run(["logits"], {k: v for k, v in feeds.items() if k in dec_in})[0][0, -1]
        logits[cfg.pad_token_id] = -np.inf
        nxt = int(logits.argmax()); out.append(nxt)
        if nxt == cfg.eos_token_id: break
    return post(tok.decode(out, skip_special_tokens=True))

g = pt.generate(**tok(src, return_tensors="pt", padding=True), num_beams=1, do_sample=False, max_length=200)
a = [post(s) for s in tok.batch_decode(g, skip_special_tokens=True)]
b = [onnx_glouton(s) for s in src]
diff = 0
for p, x, y in zip(phrases, a, b):
    print(f"{'=' if x == y else '≠'} {p}\n   PyTorch : {x}\n   ONNX q8 : {y}")
    diff += x != y
print(f"\n{len(phrases) - diff}/{len(phrases)} identiques")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Entrées encodeur : {'attention_mask', 'input_ids'} | décodeur : {'encoder_hidden_states', 'input_ids', 'encoder_attention_mask'}
= Consulter mon solde
   PyTorch : ésíɛ pfwɔ̀' nkáb wɔ́ɔn
   ONNX q8 : ésíɛ pfwɔ̀' nkáb wɔ́ɔn
= Votre solde est de {MONTANT}
   PyTorch : pfwɔ̀' nkáb gẅi ŋ́gwɔ́ 9001
   ONNX q8 : pfwɔ̀' nkáb gẅi ŋ́gwɔ́ 9001
= Frais de transfert : {FRAIS}
   PyTorch : mekǔ leńtúm : 9001
   ONNX q8 : mekǔ leńtúm : 9001
= Saisissez le numéro de téléphone du bénéficiaire
   PyTorch : ńŋwa' number kẅíkẅíle ngàŋ nkʉ̀.
   ONNX q8 : ńŋwa' number kẅíkẅíle ngàŋ nkʉ̀.
= Vous allez envoyer {MONTANT} à {NOM}
   PyTorch : Pi ge ńtúm 9001 mbwó 9112.
   ONNX q8 : Pi ge ńtúm 9001 mbwó 9112.
= Confirmer
   PyTorch : ńzẅíŋ
   ONNX q8 : ńzẅíŋ
= Annuler
   PyTorch : ḿbyág
   ONNX q8 : ḿbyág
= Transfert réussi
   PyTorch : leńtúm éfele
   ONNX q8 : leńtúm éfele
= Solde insuffisant
   PyTorch : nkáb te ńku'ku' wɔ́
   ONNX q8 : nkáb te ńku'ku' wɔ́
= Code PIN incorrect
   PyTorch : code PIN te ŋ́gwɔ́

In [20]:
import os, glob, shutil, zipfile
from google.colab import files

# 1. Téléverser la partie texte (sélectionnez paquet_ngiemboon_texte.zip)
up = files.upload()
z = [f for f in up if f.startswith("paquet_ngiemboon_texte")]
assert z, "Vous n'avez pas sélectionné paquet_ngiemboon_texte.zip"

# 2. Repartir de zéro (supprime l'ancien paquet sans texte)
shutil.rmtree("/content/paquet_ngiemboon", ignore_errors=True)
if os.path.exists("/content/paquet_ngiemboon.zip"): os.remove("/content/paquet_ngiemboon.zip")
zipfile.ZipFile(f"/content/{z[0]}").extractall("/content/")
assert os.path.exists("/content/paquet_ngiemboon/memoire/ngiemboon.json"), "La mémoire n'est pas dans le zip"

# 3. Ajouter le modèle
M = "/content/paquet_ngiemboon/modeles/ngiemboon"; os.makedirs(f"{M}/onnx", exist_ok=True)
for f in glob.glob("/content/onnx_fr_nnh/*"):
    if os.path.isfile(f): shutil.copy(f, M)
for f in glob.glob("/content/onnx_fr_nnh/onnx/*_quantized.onnx"): shutil.copy(f, f"{M}/onnx/")

# 4. tokenizer.json (si ça échoue, le paquet est quand même construit, sans le modèle utilisable)
try:
    from huggingface_hub import hf_hub_download
    from tokenizers import Tokenizer
    from transformers import MarianTokenizer
    shutil.copy(hf_hub_download("Xenova/opus-mt-fr-en", "tokenizer.json"), M)
    tj, mt = Tokenizer.from_file(f"{M}/tokenizer.json"), MarianTokenizer.from_pretrained(M)
    essais = ["Votre solde est de 9001 FCFA.", "Saisissez le numéro de téléphone du bénéficiaire",
              "Ne communiquez jamais votre code PIN à personne"]
    print("tokenizer.json compatible :", all(tj.encode(s).ids == mt(s).input_ids for s in essais))
except Exception as e:
    print("⚠ tokenizer.json non ajouté :", e)

# 5. Vérifier le contenu AVANT de télécharger
for racine, _, fichiers in os.walk("/content/paquet_ngiemboon"):
    for f in fichiers:
        p = os.path.join(racine, f); print(f"{os.path.getsize(p)/1e6:8.2f} Mo  {p.replace('/content/', '')}")

!cd /content && zip -qr paquet_ngiemboon.zip paquet_ngiemboon && du -sh paquet_ngiemboon.zip
shutil.copy("/content/paquet_ngiemboon.zip", "/content/drive/MyDrive/afroluo_models_nnh/")
files.download("/content/paquet_ngiemboon.zip")

Saving paquet_ngiemboon_texte.zip to paquet_ngiemboon_texte.zip


tokenizer.json:   0%|          | 0.00/5.64M [00:00<?, ?B/s]

PanicException: Precompiled: Error("invalid type: null, expected a borrowed string", line: 1, column: 28)

In [21]:
import os, shutil
from google.colab import files
assert os.path.exists("/content/paquet_ngiemboon/memoire/ngiemboon.json"), "Mémoire absente"
assert os.path.exists("/content/paquet_ngiemboon/modeles/ngiemboon/tokenizer.json"), "tokenizer.json absent"
for racine, _, fichiers in os.walk("/content/paquet_ngiemboon"):
    for f in fichiers:
        p = os.path.join(racine, f); print(f"{os.path.getsize(p)/1e6:8.2f} Mo  {p.replace('/content/', '')}")
if os.path.exists("/content/paquet_ngiemboon.zip"): os.remove("/content/paquet_ngiemboon.zip")
!cd /content && zip -qr paquet_ngiemboon.zip paquet_ngiemboon && du -sh paquet_ngiemboon.zip
shutil.copy("/content/paquet_ngiemboon.zip", "/content/drive/MyDrive/afroluo_models_nnh/")
files.download("/content/paquet_ngiemboon.zip")

    0.06 Mo  paquet_ngiemboon/A_FAIRE_VALIDER_PAR_LE_LOCUTEUR.csv
    0.00 Mo  paquet_ngiemboon/langue_ngiemboon.json
    0.00 Mo  paquet_ngiemboon/README_INTEGRATEUR.md
    0.05 Mo  paquet_ngiemboon/memoire/ngiemboon.json
    0.00 Mo  paquet_ngiemboon/modeles/ngiemboon/config.json
    5.64 Mo  paquet_ngiemboon/modeles/ngiemboon/tokenizer.json
    0.00 Mo  paquet_ngiemboon/modeles/ngiemboon/generation_config.json
    0.78 Mo  paquet_ngiemboon/modeles/ngiemboon/target.spm
    0.00 Mo  paquet_ngiemboon/modeles/ngiemboon/translit.py
    0.00 Mo  paquet_ngiemboon/modeles/ngiemboon/decoderNnh.js
    0.00 Mo  paquet_ngiemboon/modeles/ngiemboon/special_tokens_map.json
    1.46 Mo  paquet_ngiemboon/modeles/ngiemboon/vocab.json
    0.80 Mo  paquet_ngiemboon/modeles/ngiemboon/source.spm
    0.00 Mo  paquet_ngiemboon/modeles/ngiemboon/tokenizer_config.json
   56.73 Mo  paquet_ngiemboon/modeles/ngiemboon/onnx/decoder_model_quantized.onnx
   57.04 Mo  paquet_ngiemboon/modeles/ngiemboon/onnx/decoder

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>